# Modelos de Transporte

Os modelos de transporte são amplamente utilizados para apoiar a tomada de decisão em problemas de distribuição logística e alocação de recursos. Sua aplicação é comum em áreas como logística, cadeias de suprimentos, distribuição industrial, planejamento de produção e gestão de estoques.

O problema clássico de transporte é estruturado a partir da existência de fontes de oferta e pontos de demanda, onde cada origem possui determinada capacidade de fornecimento e cada destino apresenta uma necessidade específica. Além disso, considera-se um custo associado ao transporte entre cada origem e destino. A solução do modelo busca encontrar a quantidade ideal a ser transportada entre os pontos, respeitando as **m** restrições de oferta $(0)$ e **n** retrições de demanda $(D)$ e otimizando a função objetivo estabelecida (por exemplo, custo total $Z$).

\begin{equation}
min Z = \sum_{i=1}^{m} \sum_{j=1}^{n} c_{ij}x_{ij}
\end{equation}

sujeito a

\begin{equation}
min Z = \sum_{j=1}^{n} x_{ij} = O_i \qquad i=1, 2, ..., m.
\end{equation}

\begin{equation}
min Z = \sum_{i=1}^{m} x_{ij} = D_j \qquad i=1, 2, ..., n.
\end{equation}

\begin{equation}
x_{ij} \geq 0
\end{equation}


Somando as **m** restrições de oferta e as **n** restrições de demanda obtém-se:

\begin{equation}
\sum_{i=1}^{n} O_i = \sum_{j=1}^{n} D_j
\end{equation}

A igualdade indica que o modelo do transporte “exige uma igualdade” entre oferta total e demanda total. Porém o algoritmo a ser apresentado também pode ser utilizado quando a oferta total não for igual a demanda total.  O equilíbrio é essencial para a aplicação dos algoritmos clássicos.

A Tabela a seguir representa um exemplo clássico de um problema de transporte balanceado, no qual três origens devem atender às demandas de três destinos, considerando as quantidades ofertadas e demandadas, bem como as variáveis de decisão associadas ao fluxo de transporte entre cada origem e destino.

| Origens      | Destino 1 $(D_1)$ | Destino 2 $(D_2)$ | Destino 3 $(D_3)$ | $\cdots$     | Destino n $(D_n)$ | Oferta $(O_i)$ |
| ------------ | ----------------- | ----------------- | ----------------- | ------------ | ----------------- | -------------- |
| Origem 1     | $x_{11}$          | $x_{12}$          | $x_{13}$          | $\cdots$     | $x_{1n}$          | $O_1$          |
| Origem 2     | $x_{21}$          | $x_{22}$          | $x_{23}$          | $\cdots$     | $x_{2n}$          | $O_2$          |
| Origem 3     | $x_{31}$          | $x_{32}$          | $x_{33}$          | $\cdots$     | $x_{3n}$          | $O_3$          |
| $\vdots$     | $\vdots$          | $\vdots$          | $\vdots$          | $\ddots$     | $\vdots$          | $\vdots$       |
| Origem m     | $x_{m1}$          | $x_{m2}$          | $x_{m3}$          | $\cdots$     | $x_{mn}$          | $O_m$          |
| **Demanda**  | **$D_1$**         | **$D_2$**         | **$D_3$**         | **$\cdots$** | **$D_n$**         |                |


## Exemplo

Considere os custos agregados a seguir, em reais por tonelada:

| Origens / Destinos | Depósito 1   | Depósito 2   | Depósito 3   | Depósito   | Oferta (t) |
| ------------------ | :----------: | :----------: | :----------: | :--------: | :--------: |
| Fábrica A          | 4            | 6            | 8            | 8          | 40         |
| Fábrica B          | 6            | 8            | 6            | 7          | 60         |
| Fábrica C          | 5            | 7            | 6            | 8          | 50         |
| **Procura**        | **20**       | **30**       | **50**       | **50**     | **100**    |

In [1]:
origens = ["Fábrica A", "Fábrica B", "Fábrica C"]

destinos = ["Depósito 1", "Depósito 2", "Depósito 3", "Depósito 4"]

oferta = {
    "Fábrica A": 40,
    "Fábrica B": 60,
    "Fábrica C": 50
}

demanda = {
    "Depósito 1": 20,
    "Depósito 2": 30,
    "Depósito 3": 50,
    "Depósito 4": 50
}

custos = {
("Fábrica A","Depósito 1"):4,
("Fábrica A","Depósito 2"):6,
("Fábrica A","Depósito 3"):8,
("Fábrica A","Depósito 4"):8,

("Fábrica B","Depósito 1"):6,
("Fábrica B","Depósito 2"):8,
("Fábrica B","Depósito 3"):6,
("Fábrica B","Depósito 4"):7,

("Fábrica C","Depósito 1"):5,
("Fábrica C","Depósito 2"):7,
("Fábrica C","Depósito 3"):6,
("Fábrica C","Depósito 4"):8
}

In [2]:
import pulp as pl
# Definir problema de minimização
modelo = pl.LpProblem("Problema_de_transporte", pl.LpMinimize)

# Decision variables
x = pl.LpVariable.dicts(
    "transporte",
    [(i,j) for i in origens for j in destinos],
    lowBound=0
)

In [3]:
# Definir função objetivo
modelo += pl.lpSum(
    custos[i,j] * x[i,j]
    for i in origens
    for j in destinos
)

In [4]:
# Restrição de oferta
for i in origens:
    modelo += pl.lpSum(
        x[i,j] for j in destinos
    ) == oferta[i]

# Restrição de demanda
for j in destinos:
    modelo += pl.lpSum(
        x[i,j] for i in origens
    ) == demanda[j]

In [5]:
# Solucionar o modelo
modelo.solve()

# Resultados
for i in origens:
    for j in destinos:
        if x[i,j].varValue > 0:
            print(f"{i} → {j}: {x[i,j].varValue}")

print("\nCusto mínimo total:", pl.value(modelo.objective))

Fábrica A → Depósito 1: 10.0
Fábrica A → Depósito 2: 30.0
Fábrica B → Depósito 3: 10.0
Fábrica B → Depósito 4: 50.0
Fábrica C → Depósito 1: 10.0
Fábrica C → Depósito 3: 40.0

Custo mínimo total: 920.0
